# The cheatsheet as a machine-readable guideline, and its link with O'FAIRe

The quick reference table of the Metadata Guidelines (the cheatsheet in `docs/`) lists the 99 metadata
attributes with their usage level (Mandatory, Recommended, Optional), their FAIR principles and their
property, with the typos of the draft corrected. This notebook:

- **I.** reads the cheatsheet into a table;
- **II.** finds where EarthPortal really stores each attribute;
- **III.** saves the result as the **machine-readable guideline** (`mapping/guideline_profile.csv`);
- **IV.** links each O'FAIRe question to the attributes it reads (`mapping/ofaire_vs_guideline.csv`).

In [1]:
# Imports
import json
import re

import pandas as pd

from config import *

# I. Reading the cheatsheet

### How the cheatsheet becomes a table

The Metadata guideline exists as a document written for humans: the quick reference table (cheatsheet) of
Report 1, in `docs/`. We do not copy it into this folder. We **read it directly**, so that there is only one
source of truth: if the cheatsheet is corrected, the next run of this notebook follows.

**What the program sees.** The cheatsheet is a LaTeX table. For Python, it is only a list of text lines,
such as these three real lines:

```
\midrule[0.9pt] \multicolumn{6}{@{}l}{\cellcolor{dtband}\textbf{\color{dtnavy}1. General}} \\ \midrule[0.9pt]
1.6 & Administrators & M & -- & -- & \propb{bpm:administeredBy} \\
1.8 & URI & M & O & F1, \added{A1} & \propb{omv:URI} \\
```

The first line opens a section (here, *1. General*). The two others are attributes, with six columns
separated by `&`:

| Column | Meaning | Example (line 1.8) |
|---|---|---|
| # | number of the attribute | `1.8` |
| Attribute | its name | `URI` |
| U | usage: **M**andatory, **R**ecommended or **O**ptional | `M` |
| FsF | level in the FAIRsFAIR minimum metadata profile | `O` |
| FAIR | FAIR sub-principles; `\added{...}` marks a sub-principle we proposed | `F1, \added{A1}` |
| Primary property | the property, inside `\propb{...}` | `omv:URI` |

A dash `--` means that the guideline gives no value.

**Two patterns to recognise the lines.** We describe each kind of line with a *regular expression*, a small
pattern language that most programming languages share (JavaScript, PHP and Java have the same one, with
very small differences):

- `SECTION` finds the text `\color{dtnavy}`, a number and a dot, and **captures** the title that follows,
  up to the closing brace: on the first line above, `General`.
- `ROW` describes a whole table row and captures its six columns:

| Piece of `ROW` | It means | Captures on line 1.8 |
|---|---|---|
| `^\s*` | the start of the line, then any spaces | |
| `(\d+\.\d+)` | digits, a dot, digits | `1.8` |
| ` & ` | the column separator, written as it is | |
| `(.+?)` | any text, as short as possible | `URI` |
| `([MRO])` | exactly one letter: M, R or O | `M` |
| `(.+?)` | any text, as short as possible | `O` |
| `(.+?)` | any text, as short as possible | `F1, \added{A1}` |
| `\\propb\{` | the text `\propb{`, written as it is | |
| `([^}]+)` | one or more characters that are not `}` | `omv:URI` |
| `\}` | the closing brace | |

Three things make the pattern readable:

1. **Parentheses capture.** Each `( ... )` is a numbered group; `r.group(1)` returns what the first one
   caught, `r.group(2)` the second, and so on.
2. **`.+?` is short on purpose.** Without the `?`, `.+` first takes as much text as it can, and a group could
   spread over two columns. With the `?`, each group stops at the first ` & `.
3. **Backslashes are doubled.** In a pattern, `\d` means "a digit". To find a real backslash, as in
   `\propb`, we write `\\`. Braces have a meaning too, so a real brace is written `\{` or `\}`. The `r` in
   front of the text (`r"..."`) tells Python to keep the backslashes exactly as typed.

**The loop.** We read the file line by line, from top to bottom:

- if the line contains a section title, we **remember** it in `section`;
- otherwise, if the line is a table row, we keep its six columns, together with the last section met.

That is how each attribute knows its section (for example *Persons and Organisations*), although the title
appears only once, above the rows. `search` looks for the title **anywhere** in the line, because it sits in
the middle of a long line, while `match` only tries at the **beginning**, where a row starts. The sign `:=`
stores the result and tests it in one step: when a pattern does not match, the result is empty and the
test is false.

**Small cleanings while reading:**

| Column | Cleaning | Example |
|---|---|---|
| `usage` | the letter is written in full | `M` becomes `Mandatory` |
| `fsf_profile`, `fair` | a dash becomes an empty text, easier to test later | `--` becomes `` |
| `fair` | `\added{...}` is removed, its content kept | `F1, \added{A1}` becomes `F1, A1` |
| `property` | `dcterms:` is written `dct:`, as the portal and O'FAIRe do | (a safety net: no row uses `dcterms:` today) |

**Result.** 99 attributes in 12 sections: 20 Mandatory, 58 Recommended and 21 Optional, as in Report 1.

**Two limits to keep in mind.**

- Removing `\added{...}` also removes the information that a sub-principle is **our proposal** and not yet
  part of the guideline. The table cannot tell them apart today.
- A pattern that does not match **says nothing**: a row written differently (a missing `\propb`, for
  instance) would be skipped silently. Today all 99 numbered rows of the cheatsheet are read.

In [2]:
# We read the cheatsheet (Report 1, a LaTeX file) line by line and keep one row per attribute.
# A table row of the cheatsheet looks like:   1.7 & Status & M & R & A2 & \propb{omv:status} \\
# A section title looks like:                 \textbf{\color{dtnavy}1. General}

# ROW recognises a table row and captures its six columns, from left to right:
#   (1) the number, e.g. 1.7        (2) the attribute name, e.g. Status
#   (3) the usage: M, R or O         (4) the FsF level        (5) the FAIR sub-principles
#   (6) the property written inside \propb{...}, e.g. omv:status
ROW = re.compile(r"^\s*(\d+\.\d+) & (.+?) & ([MRO]) & (.+?) & (.+?) & \\propb\{([^}]+)\}")

# SECTION recognises a section title and captures its name, e.g. General
SECTION = re.compile(r"\\color\{dtnavy\}\d+\. ([^}]+)\}")

# The usage letters of the cheatsheet, written in full
USAGE = {"M": "Mandatory", "R": "Recommended", "O": "Optional"}

entries = []        # one dictionary per attribute, filled by the loop below
section = ""        # the section we are in; it changes each time a title is met

for line in CHEATSHEET.read_text(encoding="utf-8").splitlines():   # each line of the .tex file, in order
    if (s := SECTION.search(line)):            # is there a section title anywhere in this line?
        section = s.group(1)                   # yes: remember it for the rows that follow
    elif (r := ROW.match(line)):               # otherwise, does the line start like a table row?
        entries.append({
            "id": r.group(1),                                        # e.g. "1.7"
            "section": section,                                      # the last title met, e.g. "General"
            "entry": r.group(2),                                     # e.g. "Status"
            "usage": USAGE[r.group(3)],                              # "M" becomes "Mandatory"
            "fsf_profile": r.group(4).replace("--", ""),             # a dash means "none": empty text
            "fair": re.sub(r"\\added\{([^}]*)\}", r"\1", r.group(5)).replace("--", ""),   # \added{A1} becomes A1
            "property": r.group(6).replace("dcterms:", "dct:"),      # same prefix as the portal and O'FAIRe
        })

guideline = pd.DataFrame(entries)              # the list of dictionaries becomes a table
print(f"{len(guideline)} attributes read from the cheatsheet")
guideline.usage.value_counts()                 # how many Mandatory, Recommended and Optional

99 attributes read from the cheatsheet


usage
Recommended    58
Optional       21
Mandatory      20
Name: count, dtype: int64

In [3]:
# Quick inspection of our Metadata Guideline table
guideline.head()

,id,section,entry,usage,fsf_profile,fair,property
0,1.1,General,Acronym,Mandatory,M,F2,mod:acronym
1,1.2,General,Name,Mandatory,M,F2,omv:name
2,1.3,General,Representation language,Mandatory,M,"I1, I3",omv:hasOntologyLanguage
3,1.4,General,Categories,Mandatory,,F2,omv:hasDomain
4,1.5,General,Groups,Optional,,R1.3,bpm:group


In [4]:
# Info
guideline.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           99 non-null     object
 1   section      99 non-null     object
 2   entry        99 non-null     object
 3   usage        99 non-null     object
 4   fsf_profile  99 non-null     object
 5   fair         99 non-null     object
 6   property     99 non-null     object
dtypes: object(7)
memory usage: 5.5+ KB


In [5]:
# If one row of the cheatsheet were written slightly differently (an extra space, a missing \propb), it would simply be skipped. 
# This cell would make the notebook check it every time:
numbered = [l for l in CHEATSHEET.read_text(encoding="utf-8").splitlines() if re.match(r"^\s*\d+\.\d+ &", l)] 
assert len(numbered) == len(guideline), f"{len(numbered) - len(guideline)} rows of the cheatsheet were not read" 
# Above, we count the lines that look like a row (a number, then &) and compare that with what ROW actually read.

# II. Where EarthPortal stores each attribute

EarthPortal stores its metadata under its own property names, mostly from MOD 1.4. An attribute of the
cheatsheet finds its portal field by the first rule that works:

1. **same**: the cheatsheet property is the portal's own property;
2. **mapping**: the portal itself lists the cheatsheet property as an equivalent (its `metadataMappings`);
3. **manual**: a short, justified list in `reference/manual_links.csv`.

### Two dictionaries built from the portal's own schema

Notebook 01 saved the description of every field that EarthPortal stores (`metadata_schema.json`): 110
fields on each **submission** (one uploaded version of an artefact) and 20 on the **ontology record**
(the entry of the catalogue). Each field is described by a small record. Here is the one for the keywords:

```
attribute:        keywords
namespace:        omv
label:            Keywords
category:         description
enforce:          ["list"]
metadataMappings: ["omv:keywords", "mod:keyword", "dcat:keyword", "schema:keywords"]
```

From these 130 descriptions we build two dictionaries. A dictionary is a lookup table: give it a name, it
returns what is stored under that name.

| Dictionary | Question it answers | Example |
|---|---|---|
| `fields` | "Does the portal have a field with this exact name?" | `omv:keywords` gives (submission, its description) |
| `mappings` | "Which portal field did the portal declare **equivalent** to this property?" | `dcat:keyword` gives (submission, `omv:keywords`) |

**The full name of a field** is its namespace, a colon and its attribute: `omv` + `keywords` gives
`omv:keywords`. About 30 fields have no namespace (`submissionId`, `pullLocation`, `administeredBy`,
`viewOf`, ...): they belong to the portal itself, and we write them with the prefix `bpm:`, as O'FAIRe and
the guideline do.

**The record is kept with each name.** The same information can live on the ontology record or on the
submission, and notebook 05 must know where to read the value. That is why each entry stores a pair:
(record, ...).

**`metadataMappings` is the portal's own dictionary of synonyms.** For each field, the portal lists the
properties of other vocabularies that mean the same thing. This is what lets us find, for example, that the
guideline's `dcat:keyword` is stored by EarthPortal as `omv:keywords`. Some names are surprising: the
portal stores *Specializes* (`mod:specializes`) under `door:explanationEvolution`. Without this list, such
links would be impossible to guess. As in Part I, `dcterms:` is written `dct:`, so the guideline's
`dct:subject` finds the portal's `dcterms:subject` (stored as `omv:hasDomain`).

**When a name appears twice, the first one wins** (`setdefault` only writes if the name is not there yet).
The submission is read first. Three cases exist today:

- `omv:hasDomain` exists on **both** records; `fields` keeps the submission. The guideline's *Categories*
  are the ones of the ontology record, so this attribute is linked by hand (see below).
- `adms:translation` is declared equivalent to two opposite fields, `schema:workTranslation` and
  `schema:translationOfWork`; `mappings` keeps the first.
- `prov:generalizationOf` is declared both for `voaf:generalizes` (submission) and for `submissions`
  (ontology record); `mappings` keeps the first.

The last two do not affect our results: the guideline names `schema:workTranslation`,
`schema:translationOfWork` and `mod:generalizes`, which are found directly.

Result: `fields` holds 129 names (130 fields, one name shared by both records) and `mappings` holds 288
equivalent properties.

**The manual links.** For six attributes neither dictionary gives the right answer, so we decided the link by
hand. Each decision is written, with its reason, in `reference/manual_links.csv`:

| Cheatsheet entry | Portal field | Why |
|---|---|---|
| Acronym | `omv:acronym` (ontology record) | the cheatsheet names `mod:acronym` (MOD 3.0) |
| Categories | `omv:hasDomain` (ontology record) | the name exists on both records; categories are on the ontology record |
| Version IRI | `owl:versionIRI` | the cheatsheet names `omv:versionIRI` |
| This ontology is a view of | `bpm:viewOf` (ontology record) | the cheatsheet names `dct:isPartOf` |
| Generally related to | `door:ontologyRelatedTo` | the cheatsheet names `mod:semanticArtefactRelation` |
| Disparate modelling with | `door:hasDisparateModelling` | the cheatsheet names `mod:hasDisparateModelling` |

Keeping these decisions in a file, not in the code, means that a curator can read, question or change them
without touching Python. The next cell uses the three sources in order: **manual** first, then **same**
name, then **mapping**.

In [6]:
# We read the portal's own metadata schema, saved by notebook 01 (Part II), and build two dictionaries.
schema = json.load(open(RAW / "metadata_schema.json", encoding="utf-8"))

fields = {}         # portal name -> (record, description of the field), e.g. "omv:keywords" -> ("submission", {...})
mappings = {}       # equivalent property -> (record, portal name),    e.g. "dcat:keyword" -> ("submission", "omv:keywords")

for record in ("submission", "ontology"):                 # the two records of the portal, the submission first
    for f in schema[record]:                              # f describes one field: attribute, namespace, label, ...
        # The full name of the field: its namespace, a colon, its attribute. A field without a namespace
        # belongs to the portal itself, written bpm: (as O'FAIRe and the guideline do)
        name = f"{f.get('namespace') or 'bpm'}:{f['attribute']}"
        fields.setdefault(name, (record, f))              # keep the first record met if a name exists twice
        for equivalent in f.get("metadataMappings") or []:            # the properties the portal declares equivalent
            mappings.setdefault(equivalent.replace("dcterms:", "dct:"),  # same prefix as the cheatsheet
                                (record, name))                       # first declaration wins if repeated

# The few links we decided by hand, each with its reason, indexed by the name of the cheatsheet entry
manual = pd.read_csv(MANUAL_LINKS).set_index("entry")
manual

,portal_object,portal_attribute,reason
entry,,,
Acronym,ontology,omv:acronym,The cheatsheet names mod:acronym (MOD 3.0); th...
Categories,ontology,omv:hasDomain,omv:hasDomain exists on both the ontology reco...
Version IRI,submission,owl:versionIRI,The cheatsheet names omv:versionIRI; the porta...
This ontology is a view of,ontology,bpm:viewOf,The cheatsheet names dct:isPartOf; the portal ...
Generally related to,submission,door:ontologyRelatedTo,The cheatsheet names mod:semanticArtefactRelat...
Disparate modelling with,submission,door:hasDisparateModelling,The cheatsheet names mod:hasDisparateModelling...


In [7]:
#Info
manual.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, Acronym to Disparate modelling with
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   portal_object     6 non-null      object
 1   portal_attribute  6 non-null      object
 2   reason            6 non-null      object
dtypes: object(3)
memory usage: 192.0+ bytes


In [8]:
print(f"{len(fields)} fields read from the portal's metadata schema")
print(f"{len(mappings)} mappings read from the portal's metadata schema")
print(f"{len(manual)} manual links defined")

129 fields read from the portal's metadata schema
288 mappings read from the portal's metadata schema
6 manual links defined


In [9]:
# How does "fields" look like?
print(f"One example of a field: {list(fields.items())[0]}")

One example of a field: ('bpm:submissionId', ('submission', {'@id': 'http://data.bioontology.org/submission_metadata/submissionId', '@type': 'http://data.bioontology.org/metadata/SubmissionMetadata', 'attribute': 'submissionId', 'namespace': None, 'label': None, 'extracted': False, 'metadataMappings': None, 'enforce': ['existence', 'integer', 'no_list'], 'enforcedValues': None, 'category': 'no', '@context': {'@vocab': 'http://data.bioontology.org/metadata/'}}))


In [10]:
# What about "mappings"?
print(f"One example of a mapping: {list(mappings.items())[0]}")

One example of a mapping: ('mod:prefLabelProperty', ('submission', 'bpm:prefLabelProperty'))


### Linking each attribute to its portal field

For each of the 99 attributes read from the cheatsheet, we now ask: **in which field does EarthPortal
really store it?** The function `portal_field` tries the three sources of the previous cell, always in the
same order, and keeps the first answer:

| Order | Rule | Question asked | Example |
|---|---|---|---|
| 1 | **manual** | Did we decide this link by hand? | *Version IRI*: `omv:versionIRI` is stored as `owl:versionIRI` |
| 2 | **same** | Does the portal have a field with exactly the cheatsheet's property? | *Name*: `omv:name` is `omv:name` |
| 3 | **mapping** | Did the portal declare this property equivalent to one of its fields? | *Keywords*: `dcat:keyword` is stored as `omv:keywords` |
| | not found | None of the three answered | (no case today) |

**Why the manual rule comes first.** A decision taken by a person must win over an automatic one. The case
of *Categories* shows it: the cheatsheet's property `omv:hasDomain` exists in the portal, so rule 2 would
answer, but with the field of the **submission**. The categories shown on EarthPortal are the ones of the
**ontology record**, so our manual link points there.

**Three new columns per attribute.** `portal_field` returns three answers at once: the record (`submission`
or `ontology`), the portal field, and the rule that found it. `apply(..., axis=1)` runs the function on
each row, and `result_type="expand"` spreads the three answers over the three columns `portal_object`,
`portal_attribute` and `link_rule`.

**Two more columns computed from them:**

| Column | How | Example | Used for |
|---|---|---|---|
| `api_key` | the part after the colon of the portal field | `omv:keywords` gives `keywords` | notebook 05 reads each value under this key in the API answers |
| `differs_from_portal` | `yes` when the cheatsheet's property is not the portal's field | `dcat:keyword` vs `omv:keywords`: `yes` | the column showing where the guideline (MOD 3.0) and the portal differ |

**Result.**

| Rule | Attributes | Stored under another property |
|---|---|---|
| same | 72 | 0 |
| mapping | 21 | 21 |
| manual | 6 | 5 |
| **total** | **99** (all linked) | **26** |

The 26 attributes stored under another property are listed below the cell. They come from the 21 mappings
and from 5 of the 6 manual links: *Categories* is linked by hand for its **record**, not for its name, so
its property does not differ.

**What to keep in mind.** The **mapping** rule trusts the synonyms that the portal declares. Most are
obvious (`dct:creator` is stored as `omv:hasCreator`), some are surprising (*Specialization of*,
`mod:specializes`, is stored as `door:explanationEvolution`; *Has equivalences with* as
`door:isAlignedTo`). They are the portal's choices, not ours, and the list below lets a curator check each
of them. If a future version of the cheatsheet or of the portal brings an attribute that none of the three
rules can link, it appears as **not found** instead of being dropped silently.

In [11]:
# We find, for each attribute of the cheatsheet, the field where EarthPortal really stores it.
# The three sources of the previous cell are tried in a fixed order; the first one that answers wins.
def portal_field(row):
    """We return (record, portal field, rule) for one attribute of the cheatsheet."""
    if row.entry in manual.index:                       # 1. a link decided by hand always has the last word
        return manual.loc[row.entry, "portal_object"], manual.loc[row.entry, "portal_attribute"], "manual"
    if row.property in fields:                          # 2. the portal has a field with exactly this name
        return fields[row.property][0], row.property, "same"
    if row.property in mappings:                        # 3. the portal declared this property equivalent to one of its fields
        return mappings[row.property][0], mappings[row.property][1], "mapping"
    return "", "", "not found"                          # none of the three: to be examined by a curator

# We apply the function to each row (axis=1) and spread the three answers over three new columns
guideline[["portal_object", "portal_attribute", "link_rule"]] = guideline.apply(portal_field, axis=1, result_type="expand")

# The key under which the API returns the value: the part after the colon, e.g. "omv:keywords" gives "keywords"
guideline["api_key"] = guideline.portal_attribute.str.split(":").str[1]

# "yes" when the guideline and the portal use two different properties for the same attribute
guideline["differs_from_portal"] = (guideline.property != guideline.portal_attribute).map({True: "yes", False: "no"})

print(guideline.link_rule.value_counts().to_dict())                       # how many attributes each rule linked
print(f"{(guideline.differs_from_portal == 'yes').sum()} attributes are stored by the portal under another property")
guideline[guideline.differs_from_portal == "yes"][["id", "entry", "usage", "property", "portal_attribute", "link_rule"]]

{'same': 72, 'mapping': 21, 'manual': 6}
26 attributes are stored by the portal under another property


,id,entry,usage,property,portal_attribute,link_rule
0,1.1,Acronym,Mandatory,mod:acronym,omv:acronym,manual
8,1.9,Version IRI,Recommended,omv:versionIRI,owl:versionIRI,manual
19,2.6,Keywords,Recommended,dcat:keyword,omv:keywords,mapping
20,2.7,Subjects,Recommended,dct:subject,omv:hasDomain,mapping
25,3.2,Modification date,Mandatory,dct:modified,omv:modificationDate,mapping
28,3.5,Submission date,Mandatory,dct:dateSubmitted,omv:creationDate,mapping
35,5.2,Creator,Mandatory,dct:creator,omv:hasCreator,mapping
36,5.3,Contributor,Recommended,dct:contributor,omv:hasContributor,mapping
40,5.7,Endorsed by,Recommended,mod:endorsedBy,omv:endorsedBy,mapping
55,8.1,Bug database,Recommended,doap:bug-database,doap:bugDatabase,mapping


# III. The machine-readable guideline

We add two pieces of information that a program needs: what kind of value the portal expects (an IRI,
a list, a date...) with its allowed values, and **who can fill the field in**. The last one is a
proposal, to be validated with the team:

- **portal**: the portal generates the value itself (download links, search endpoints, submission date);
- **curator**: the EarthPortal team sets it (categories, groups, visibility, administrators, curation);
- **author**: everything else, filled in the file or in the submission form by the producer.

### From a table to a guideline that a program can use

After Part II, each of the 99 attributes knows **where** the portal stores it. To check a metadata record
automatically, a program needs two more things for each attribute: **who** can fill it in, and **what kind of
value** the portal expects.

**Who can fill it in (`who_fixes`).** We sort the attributes into three groups:

| Group | Meaning | Attributes |
|---|---|---|
| **portal** | the portal generates the value itself; an empty value means that processing did not produce it | Download URL, CSV dump, URI lookup endpoint, Free-text search endpoint, Submission date |
| **curator** | the EarthPortal team sets the value | Categories, Groups, Administrators, Visibility, This ontology is a view of, Indexed in catalog, Curator, Curation date |
| **author** | the producer of the artefact, in the file or in the submission form | the 86 others |

*Categories* is recognised by its name and not by its field, because its field (`omv:hasDomain`) also
exists on the submission, where the author fills it in.

| who_fixes | Mandatory | Recommended | Optional |
|---|---|---|---|
| author | 12 | 54 | 20 |
| curator | 3 | 4 | 1 |
| portal | 5 | 0 | 0 |

This split is **our proposal**: it must be validated with the EarthPortal team before anyone is contacted.

**What the portal expects (`portal_expects`, `allowed_values`).** The portal's own schema describes each field
with short codes, kept as they are:

- `uri` (the value must be an IRI), `list` (several values are allowed), `date_time`, `boolean`, `Agent` (a
  person or an organisation record), and a few others used by the portal's code (`existence`, `unique`, ...);
  `no_list` only says "a single value", so we leave it out;
- for 12 attributes, a **list of allowed values**. When the portal gives them as pairs (IRI, label), we keep
  the IRIs, which are what a program compares. For example:

| Attribute | Allowed values |
|---|---|
| Status | `alpha`, `beta`, `production`, `retired` |
| License | 9 licence IRIs, e.g. `https://creativecommons.org/licenses/by/4.0/` |
| Syntax | 19 format IRIs, e.g. `http://www.w3.org/ns/formats/N3` |
| Indexed or Included in catalog or repository | 120 catalogues |

The allowed values make value checks possible, not only presence checks: an artefact that declares the
syntax `https://www.w3.org/ns/formats/data/N3` uses a value that is **not** in the portal's own list. Some
IRIs of the portal's schema end with a space; we remove it (`strip`) so that comparisons work.

A field name can exist on both records (`omv:hasDomain`), so `expects` looks for the field with the **same
record and the same name** before reading its codes.

**The result: the machine-readable guideline.** It is saved twice in `mapping/`:

- `guideline_profile.csv`: one row per attribute, 15 columns, for people and spreadsheets; notebook 05 reads
  this one;
- `guideline_profile.json`: the same content as a list of records, for programs and web pages. One record
  looks like this:

```json
{
 "id": "1.7",
 "section": "General",
 "entry": "Status",
 "usage": "Mandatory",
 "fsf_profile": "R",
 "fair": "A2",
 "property": "omv:status",
 "portal_object": "submission",
 "portal_attribute": "omv:status",
 "link_rule": "same",
 "api_key": "status",
 "differs_from_portal": "no",
 "who_fixes": "author",
 "portal_expects": "existence",
 "allowed_values": "alpha | beta | production | retired"
}
```

With this file, checking the Mandatory attributes of a record takes three steps: keep the rows where `usage`
is Mandatory, read each value under its `api_key` in the right record (`portal_object`), and test that it
is not empty (and, when `allowed_values` is given, that it is one of them). Notebook 05 does it for every
ontology of the catalogue.

In [12]:
# Who can fill each attribute in? A proposal, to be validated with the EarthPortal team.
# Values that the portal generates itself (download links, endpoints, submission date):
PORTAL = {"bpm:csvDump", "void:dataDump", "void:uriLookupEndpoint", "void:openSearchDescription", "omv:creationDate"}
# Values that the EarthPortal team sets (administrators, groups, visibility, views, catalogues, curation):
CURATOR = {"bpm:administeredBy", "bpm:group", "bpm:viewingRestriction", "bpm:viewOf",
           "schema:includedInDataCatalog", "pav:curatedBy", "pav:curatedOn"}


def who_fixes(row):
    """We return portal, curator or author for one attribute of the cheatsheet."""
    if row.portal_attribute in PORTAL:                  # generated by the portal
        return "portal"
    if row.portal_attribute in CURATOR or (row.entry == "Categories"):   # set by the team; Categories by name,
        return "curator"                                                 # since omv:hasDomain also exists on the submission
    return "author"                                     # everything else: the producer, in the file or the form


def expects(row):
    """We return what the portal expects for one attribute: the kind of value, and its allowed values."""
    # A first guess from the dictionary of fields (it keeps the first record met for a given name)
    f = fields.get(row.portal_attribute, (None, {}))[1] if row.portal_object else {}
    # The exact field: same record AND same name, since one name can exist on both records
    for record in ("submission", "ontology"):
        for g in schema[record]:
            if record == row.portal_object and f"{g.get('namespace') or 'bpm'}:{g['attribute']}" == row.portal_attribute:
                f = g
    allowed = f.get("enforcedValues") or []                         # the allowed values, if the portal has a list
    allowed = list(allowed) if isinstance(allowed, dict) else allowed   # a dictionary {IRI: label}: we keep the IRIs
    kind = ", ".join(e for e in f.get("enforce") or [] if e != "no_list")   # e.g. "uri, list"; "no_list" says nothing
    return kind, " | ".join(str(a).strip() for a in allowed)        # e.g. "alpha | beta | production | retired"


guideline["who_fixes"] = guideline.apply(who_fixes, axis=1)                         # one new column
guideline[["portal_expects", "allowed_values"]] = guideline.apply(expects, axis=1, result_type="expand")   # two new columns

# The machine-readable guideline, twice: a table (CSV) for people and spreadsheets, a JSON file for programs
guideline.to_csv(MAPPING / "guideline_profile.csv", index=False)
guideline.to_json(MAPPING / "guideline_profile.json", orient="records", indent=1, force_ascii=False)
print(f"Saved {(MAPPING / 'guideline_profile.csv').relative_to(REPO)} and its JSON twin")
pd.crosstab(guideline.who_fixes, guideline.usage)          # how many attributes each actor can fill, by usage

Saved earthportal_sa_audit/mapping/guideline_profile.csv and its JSON twin


usage,Mandatory,Optional,Recommended
who_fixes,,,
author,12,20,54
curator,3,1,4
portal,5,0,0


**Conclusion**: `guideline_profile.csv` is the guideline in a form a program can use. For instance, the
Mandatory attributes of one record can be checked in three lines; notebook 05 does it for every ontology.

# IV. Which attributes each O'FAIRe question reads

O'FAIRe names the fields with MOD 1.4 properties and reads each one from the API by its plain name
(`reference/ofaire_properties.config.json`, taken from the O'FAIRe source code). Each O'FAIRe result
lists the properties of its question. This list comes from O'FAIRe's configuration, and the code that
gives the score reads the same properties except in a few places, kept with their evidence in
`reference/ofaire_code_vs_config.csv`. We apply them, so that every question is attached to the
cheatsheet attributes the code really reads.

### The meeting point of three vocabularies

Notebook 05 turns the points an ontology loses into actions ("fill in *Version IRI*"). For that, it must know
**which attributes of the guideline each O'FAIRe question reads**. This is not obvious, because the same
piece of metadata has three names:

| World | Name of the subjects of an artefact |
|---|---|
| O'FAIRe (MOD 1.4) | `dct:subject` |
| EarthPortal | the field `hasDomain` on the submission |
| The guideline (cheatsheet) | *Subjects* (2.7), property `dct:subject`, stored as `omv:hasDomain` |

The three names meet in **one place**: the record and the API key under which the portal returns the value,
here (submission, `hasDomain`). This cell builds every such link in four steps.

**1. O'FAIRe's own table of properties.** O'FAIRe's source code contains a table
(`reference/ofaire_properties.config.json`, 123 entries) that says, for each property it uses, where the portal
keeps the value (`source`) and under which name:

| source | entries | Example |
|---|---|---|
| submission | 99 | `dct:language` is read as `naturalLanguage` |
| ontology | 5 | `dct:title` is read as `name` |
| metrics | 10 | `omv:numberOfClasses` is read as `classes` |
| link | 9 | `void:rootResource` is read as `roots` |

From it we build the dictionary `ofaire_fields`: property gives (source, API key).

**2. The properties of each question.** Each O'FAIRe result lists, for each question, the properties it shows.
We collect them for the 61 questions. The lists are the same for every ontology; we still join them over all
results (a set keeps each property once), so that nothing is missed if one day they differ.

**3. The corrections from the Java code.** The lists shown in the results come from O'FAIRe's configuration,
and **they are for display only**: the Java code decides what is really read. When we compared both (commit
`24def6b`), four lines differed. We follow the code, and each correction is written with its evidence in
`reference/ofaire_code_vs_config.csv`:

| Question | Change | Why | Evidence |
|---|---|---|---|
| F2Q1 | + `schema:translator` | the code also counts the translator | `F2.java`, line 29 |
| I3Q3 | + `dct:language` | the code also tests the natural language | `I3.java`, line 59 |
| I3Q3 | − `dct:subject` | the code asks for a field named `subject`, which does not exist (the portal stores `hasDomain`), so it never counts | `I3.java`, line 58 |
| R1.3Q2 | + `omv:acronym` | the code compares the acronym with six OBO Foundry acronyms | `R13.java`, lines 51 and 54 |

**4. The join.** On the guideline side, each attribute already knows its (record, API key) since Part II. We
build `by_field`, a dictionary indexed by this pair, and for every property of every question we look up:
property, then (source, API key) through `ofaire_fields`, then the cheatsheet attribute through `by_field`.
For example: F1Q4 reads `owl:versionIRI`, which O'FAIRe finds at (submission, `versionIRI`), where the
guideline's attribute 1.9 *Version IRI* is stored.

Each row also says **who can act** on the property:

- the cheatsheet's `who_fixes` when the attribute is in the cheatsheet;
- **tool or repository** when the question reads no property at all (23 questions: the 12 that O'FAIRe gives
  to everybody, the 10 that are not implemented, and F3Q1);
- **portal** for the metrics and links that the portal computes (number of classes, roots, analytics, ...);
- **author** for the other fields that the guideline does not describe (the object description properties
  such as `prefLabelProperty`, `owl:versionInfo`, `owl:deprecated`).

**Result** (`mapping/ofaire_vs_guideline.csv`, one row per question and property): 181 rows, 61 questions,
**158 properties read**, of which **33 have no entry in the cheatsheet**.

| who_fixes | rows |
|---|---|
| author | 126 |
| portal | 23 |
| tool or repository | 23 |
| curator | 9 |

The 33 properties without a cheatsheet entry are also useful for the guideline itself: they show what
O'FAIRe measures and the guideline does not describe yet (version information, deprecation, the object
description properties). They are a concrete input for the discussion on the next version of the
Metadata guideline.

In [13]:
# I. O'FAIRe's own table of properties (reference/ofaire_properties.config.json, copied from its source code):
#    for each MOD 1.4 property, where the portal keeps the value (source) and under which API key.
ofaire_fields = {}          # O'FAIRe property -> (source, API key), e.g. "dct:subject" -> ("submission", "hasDomain")
for p in json.load(open(OFAIRE_PROPERTIES, encoding="utf-8"))["properties"]:
    # A metric (number of classes, ...) is read under its own key; any other field under its portal name
    name = p["key"] if p["source"] == "metrics" and p.get("key") else p["portal"].split(":")[1]
    ofaire_fields.setdefault(p["mod"].replace(";", ":"), (p["source"], name))   # ";" for ":" guards against a typo

# II. The properties of each question, as the O'FAIRe results list them
ofaire = json.load(open(RAW / "ofaire.json", encoding="utf-8"))
reads = {}                  # question -> {"max_credits": ..., "properties": {...}}
for result in ofaire.values():                                     # one result per ontology
    for principle in ("Findable", "Accessible", "Interoperable", "Reusable"):
        for block in result[principle].values():                   # the sub-principles (and a few numbers, skipped)
            if isinstance(block, dict):
                for qid, q in block.get("results", {}).items():    # the questions of this sub-principle
                    reads.setdefault(qid, {"max_credits": q["maxCredits"], "properties": set()})
                    reads[qid]["properties"] |= set((q.get("properties") or {}).keys())   # add them to the set

# III. Where the Java code reads something else than the configuration declares, we follow the code
code_vs_config = pd.read_csv(CODE_VS_CONFIG)                       # 4 rows, each with its evidence
for c in code_vs_config.itertuples():
    if c.change == "add":
        reads[c.question]["properties"].add(c.property)            # the code reads it, the list forgot it
    else:
        reads[c.question]["properties"].discard(c.property)        # the list shows it, the code never counts it

# IV. The meeting point: O'FAIRe gives (source, API key), the guideline gives (record, API key)
by_field = {(r.portal_object, r.api_key): r for r in guideline.itertuples()}
rows = []
for qid, info in reads.items():
    for prop in sorted(info["properties"]) or [""]:                # a question that reads nothing still gets one row
        source, name = ofaire_fields.get(prop, ("", ""))          # where O'FAIRe reads this property
        g = by_field.get((source, name))                          # the cheatsheet attribute stored there, if any
        # Who can act: the cheatsheet's answer if the attribute is there; otherwise nobody (no property),
        # the portal (metrics and links it computes), or the author (fields the guideline does not describe)
        who = g.who_fixes if g else ("tool or repository" if not prop else "portal" if source in ("metrics", "link") else "author")
        rows.append({"question": qid, "max_credits": info["max_credits"], "ofaire_property": prop,
                     "api_key": name, "cheatsheet_id": g.id if g else "", "cheatsheet_entry": g.entry if g else "",
                     "usage": g.usage if g else "", "who_fixes": who})
ofaire_vs_guideline = pd.DataFrame(rows)                           # one row per (question, property)
ofaire_vs_guideline.to_csv(MAPPING / "ofaire_vs_guideline.csv", index=False)

# The properties O'FAIRe reads that have no entry in the cheatsheet
not_in_cheatsheet = ofaire_vs_guideline[(ofaire_vs_guideline.ofaire_property != "") & (ofaire_vs_guideline.cheatsheet_entry == "")]
print(f"{ofaire_vs_guideline.question.nunique()} questions, {(ofaire_vs_guideline.ofaire_property != '').sum()} properties read; "
      f"{len(not_in_cheatsheet)} of them have no entry in the cheatsheet")
ofaire_vs_guideline.head(10)

61 questions, 158 properties read; 33 of them have no entry in the cheatsheet


,question,max_credits,ofaire_property,api_key,cheatsheet_id,cheatsheet_entry,usage,who_fixes
0,F1Q1,9.0,owl:ontologyIRI,URI,1.8,URI,Mandatory,author
1,F1Q2,11.0,dct:identifier,identifier,1.14,Other identifier,Optional,author
2,F1Q3,12.0,,,,,,tool or repository
3,F1Q4,9.0,owl:versionIRI,versionIRI,1.9,Version IRI,Recommended,author
4,F2Q1,16.0,dct:alternative,alternative,2.9,Alternative name,Recommended,author
5,F2Q1,16.0,dct:audience,audience,8.2,Audience,Recommended,author
6,F2Q1,16.0,dct:coverage,coverage,9.3,Coverage,Recommended,author
7,F2Q1,16.0,dct:description,description,2.2,Description,Mandatory,author
8,F2Q1,16.0,dct:publisher,publisher,5.5,Publisher,Recommended,author
9,F2Q1,16.0,dct:subject,hasDomain,2.7,Subjects,Recommended,author


**Conclusion**: most properties read by O'FAIRe have an entry in the cheatsheet. Those without one are
either computed by the portal (metrics, links) or fields the guideline chose not to describe (the object
description properties, the version information, the deprecation flag); they appear in the table above
with an empty `cheatsheet_entry`.